# Marketplace Listing Automation System

**Python • Selenium • Pandas • Excel Automation**

This notebook demonstrates a data-driven browser automation workflow for creating Facebook Marketplace listings.

### What the system does

1. Reads product titles and prices from an Excel workbook.
2. Matches product numbers with local image files.
3. Opens a Chrome browser through Selenium.
4. Lets the user complete Facebook authentication manually.
5. Uploads product images and fills listing fields.
6. Selects category and condition.
7. Adds a reusable description and product tags.
8. Publishes listings in a controlled batch.

> **Portfolio note:** This notebook is intentionally organized as a reproducible project rather than as a single long script. Website interfaces and selectors can change over time, so the automation functions may require updates when the Marketplace UI changes.

### Project workflow

**Excel Inventory → Data Validation → Selenium Browser → Listing Form → Publish**


## 1. Expected Project Files

For the minimal GitHub repository, keep the notebook together with the input workbook and image files used for testing.

```text
facebook-marketplace-automation/
│
├── Marketplace_Automation.ipynb
├── README.md
├── requirements.txt
├── workflow.png
├── items.xlsx
├── 1.jpg
├── 2.jpg
└── ...
```

For a public repository, use **sample/non-sensitive data** only. Do not commit real customer information, account credentials, browser profiles, CAPTCHA images, or private inventory data.


## 2. Environment Setup

Install the required packages once before running the notebook.

In [ ]:
# Run this cell only when setting up a new environment.
# %pip install selenium webdriver-manager pandas openpyxl pillow matplotlib


### Import Libraries

The original project uses Selenium for browser interaction, Pandas for spreadsheet data, Pillow/Matplotlib for image inspection, and WebDriver Manager for ChromeDriver setup. fileciteturn1file0L1-L21


In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementNotInteractableException,
    TimeoutException,
    NoSuchElementException,
    WebDriverException,
)
from webdriver_manager.chrome import ChromeDriverManager

print("Environment imports loaded successfully.")


## 3. Project Configuration

Change the paths and listing defaults here instead of editing the automation functions.

In [ ]:
# -----------------------------
# File / folder configuration
# -----------------------------
EXCEL_PATH = Path("items.xlsx")

# Listing images are named 1.jpg, 2.jpg, 3.jpg, ...
IMAGE_EXTENSIONS = ["jpeg", "jpg", "png", "gif", "bmp", "tiff", "webp"]

# -----------------------------
# Marketplace defaults
# -----------------------------
MARKETPLACE_CREATE_URL = "https://www.facebook.com/marketplace/create/item"

DEFAULT_CATEGORY = "Furniture"
DEFAULT_CONDITION = "New"

DEFAULT_DESCRIPTION = (
    "Mattresses For Sale (Single, Double, Queen, King).\n"
    "All sizes available:\n"
    "✓ Single/Twin: 38×75 inches\n"
    "✓ Double/Full: 54×75 inches\n"
    "✓ Queen: 60×80 inches\n"
    "✓ King: 76×80 inches\n"
    "Cash on Delivery | Home Delivery Available.\n"
    "Message for pricing details!"
)

DEFAULT_TAGS = [
    "Bed",
    "Mattress",
    "Sofa",
    "King Size Bed",
    "Queen Size Bed",
]

# Optional persistent Chrome profile.
# Keep this folder OUT of GitHub.
CHROME_PROFILE_DIR = Path("chrome_profile")

print(f"Excel source: {EXCEL_PATH}")
print(f"Default category: {DEFAULT_CATEGORY}")
print(f"Default condition: {DEFAULT_CONDITION}")


## 4. Load and Validate Inventory Data

The source project expects an Excel file with at least two columns: `Title` and `price`. fileciteturn1file0L27-L38

The function below adds basic validation so a missing file or missing required columns is detected before browser automation starts.


In [ ]:
def load_data(excel_path=EXCEL_PATH):
    """Load and validate product titles and prices from Excel."""
    excel_path = Path(excel_path)

    if not excel_path.exists():
        raise FileNotFoundError(
            f"Data file '{excel_path}' was not found. "
            "Place the workbook beside this notebook or update EXCEL_PATH."
        )

    df = pd.read_excel(excel_path)

    required_columns = {"Title", "price"}
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"Missing required Excel columns: {sorted(missing)}. "
            f"Found columns: {list(df.columns)}"
        )

    df = df[["Title", "price"]].copy()
    df["Title"] = df["Title"].astype(str).str.strip()
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    if df["Title"].eq("").any():
        raise ValueError("One or more product titles are empty.")

    if df["price"].isna().any():
        bad_rows = df.index[df["price"].isna()].tolist()
        raise ValueError(f"Invalid price values found in rows: {bad_rows}")

    return df.reset_index(drop=True)


inventory = load_data()
inventory.head()


## 5. Match Product Images

The current workflow maps listing numbers to files such as `1.jpg`, `2.jpg`, and `3.jpg`, checking several common image extensions. fileciteturn1file0L41-L57


In [ ]:
def get_image_paths(num_listings, image_extensions=IMAGE_EXTENSIONS):
    """Find an image for each listing number and return absolute paths."""
    paths = []

    for i in range(1, num_listings + 1):
        found_path = None

        for ext in image_extensions:
            candidate = Path(f"{i}.{ext}")
            if candidate.exists():
                found_path = candidate.resolve()
                break

        if found_path is None:
            print(f"Warning: no image found for listing {i}.")
        else:
            paths.append(found_path)

    return paths


def preview_image(image_path):
    """Display a local product image for a quick visual check."""
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = Image.open(image_path)
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(image_path.name)
    plt.show()


print("Image matching functions are ready.")


## 6. Select the Batch to Process

Run the following cell after checking the inventory. For a first test, start with **one listing** rather than a full batch.


In [ ]:
MAX_AVAILABLE = len(inventory)

# Change this value for your test.
NUM_LISTINGS = min(1, MAX_AVAILABLE)

if NUM_LISTINGS < 1:
    raise ValueError("No inventory records are available.")

selected_inventory = inventory.iloc[:NUM_LISTINGS].copy()
image_paths = get_image_paths(NUM_LISTINGS)

print(f"Inventory rows selected: {len(selected_inventory)}")
print(f"Images found: {len(image_paths)}")

if len(image_paths) != NUM_LISTINGS:
    raise ValueError(
        "The number of matching images does not equal the selected listing count. "
        "Add the missing image files before continuing."
    )

selected_inventory


## 7. Start Chrome

The browser setup uses Selenium and WebDriver Manager, matching the original project's approach. fileciteturn1file0L60-L85

A separate local Chrome profile can be used so the browser session does not have to be recreated every time. **Never upload that profile folder to GitHub.**


In [ ]:
def open_chrome(profile_dir=CHROME_PROFILE_DIR):
    """Create and return a Chrome WebDriver instance."""
    profile_dir = Path(profile_dir).resolve()
    profile_dir.mkdir(parents=True, exist_ok=True)

    chrome_options = Options()
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument(f"--user-data-dir={profile_dir}")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    return driver


driver = open_chrome()
print("Chrome started successfully.")


## 8. Facebook Authentication

For the public portfolio version, credentials are **not stored in code**.

The safer notebook workflow is:

1. Open Marketplace in the automated browser.
2. Complete Facebook login yourself in the browser.
3. Complete any security verification/CAPTCHA yourself if Facebook asks for it.
4. Return to the notebook and continue.

This avoids publishing passwords and keeps the authentication step under the account owner's control.


In [ ]:
def open_marketplace(driver):
    """Open the Marketplace listing creation page."""
    driver.get(MARKETPLACE_CREATE_URL)
    time.sleep(4)
    print("Marketplace page opened.")
    print("Complete Facebook login/security verification in Chrome if required.")
    input("Press ENTER here after you are ready to continue...")


open_marketplace(driver)


## 9. Marketplace Form Automation

The following reusable functions represent the core automation steps from the original script: image upload, title, price, category, condition, description, tags, next, and publish. fileciteturn1file0L171-L277


In [ ]:
def wait_click(driver, by, value, timeout=15):
    element = WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, value))
    )
    element.click()
    return element


def upload_image(driver, image_path):
    """Upload a single listing image."""
    image_path = Path(image_path).resolve()

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    file_input = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "input[type='file']"))
    )
    file_input.send_keys(str(image_path))
    time.sleep(3)


def enter_title(driver, title):
    """Enter the listing title."""
    title_input = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located(
            (By.XPATH, "//label[@aria-label='Title']//input")
        )
    )
    title_input.clear()
    title_input.send_keys(str(title))


def enter_price(driver, price):
    """Enter the listing price."""
    price_input = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located(
            (By.XPATH, "//label[@aria-label='Price']//input")
        )
    )
    price_input.clear()
    price_input.send_keys(str(price))


def select_category(driver, category_name=DEFAULT_CATEGORY):
    """Select a Marketplace category."""
    wait_click(driver, By.XPATH, "//label[@aria-label='Category']")
    time.sleep(1)

    wait_click(
        driver,
        By.XPATH,
        f"//span[normalize-space()='{category_name}']"
    )
    time.sleep(1)


def select_condition(driver, condition_name=DEFAULT_CONDITION):
    """Select item condition."""
    wait_click(driver, By.XPATH, "//label[@aria-label='Condition']")
    time.sleep(1)

    wait_click(
        driver,
        By.XPATH,
        f"//span[normalize-space()='{condition_name}']"
    )
    time.sleep(1)


def enter_description(driver, description=DEFAULT_DESCRIPTION):
    """Enter the product description."""
    desc_input = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located(
            (By.XPATH, "//label[@aria-label='Description']//textarea")
        )
    )
    desc_input.clear()
    desc_input.send_keys(description)


def add_tags(driver, tags=None):
    """Add product tags when the Marketplace field is available."""
    tags = tags or DEFAULT_TAGS

    try:
        tags_input = WebDriverWait(driver, 7).until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    "//label[@aria-label='Product tags']//textarea"
                    " | //label[@aria-label='Product tags']//input",
                )
            )
        )

        for tag in tags:
            tags_input.send_keys(tag)
            tags_input.send_keys(Keys.ENTER)
            time.sleep(0.4)

    except TimeoutException:
        print("Product tags field was not displayed; continuing without tags.")


def click_next(driver):
    """Move to the next step of the listing workflow."""
    wait_click(
        driver,
        By.XPATH,
        "//div[@aria-label='Next'] | //span[normalize-space()='Next']"
    )
    time.sleep(3)


def click_publish(driver):
    """Publish the current listing."""
    wait_click(
        driver,
        By.XPATH,
        "//div[@aria-label='Publish'] | //span[normalize-space()='Publish']"
    )
    time.sleep(5)


print("Marketplace form functions loaded.")


## 10. Build One Listing

Keeping the workflow in one function makes the automation easier to understand, test, and maintain.


In [ ]:
def create_one_listing(
    driver,
    title,
    price,
    image_path,
    category=DEFAULT_CATEGORY,
    condition=DEFAULT_CONDITION,
    description=DEFAULT_DESCRIPTION,
    tags=None,
):
    """Create and publish a single Marketplace listing."""
    driver.get(MARKETPLACE_CREATE_URL)
    time.sleep(4)

    upload_image(driver, image_path)
    enter_title(driver, title)
    enter_price(driver, price)
    select_category(driver, category)
    select_condition(driver, condition)
    enter_description(driver, description)
    add_tags(driver, tags)

    click_next(driver)
    click_publish(driver)

    return True


## 11. Batch Processing

The original program loops through the available products and reports errors per listing. fileciteturn1file0L280-L305

This version keeps a structured result table, which is more useful for debugging and portfolio demonstration.


In [ ]:
def create_facebook_listings(driver, inventory_df, image_paths):
    """Process a batch of listings and return a results DataFrame."""
    total = min(len(inventory_df), len(image_paths))
    results = []

    for i in range(total):
        row = inventory_df.iloc[i]
        print(f"\n--- Listing {i + 1}/{total}: {row['Title']} ---")

        try:
            create_one_listing(
                driver=driver,
                title=row["Title"],
                price=row["price"],
                image_path=image_paths[i],
            )

            results.append({
                "listing_number": i + 1,
                "title": row["Title"],
                "price": row["price"],
                "status": "Success",
                "error": "",
            })

            print(f"Success: listing {i + 1} published.")

        except Exception as exc:
            results.append({
                "listing_number": i + 1,
                "title": row["Title"],
                "price": row["price"],
                "status": "Failed",
                "error": str(exc),
            })

            print(f"Failed: listing {i + 1} -> {exc}")

    return pd.DataFrame(results)


## 12. Preflight Check

Run this before publishing anything. It checks the basic data/image alignment without opening additional listings.


In [ ]:
def preflight_check(inventory_df, image_paths):
    """Validate that the selected batch is ready for automation."""
    checks = {
        "inventory_rows": len(inventory_df),
        "image_files": len(image_paths),
        "titles_present": inventory_df["Title"].notna().all(),
        "prices_valid": inventory_df["price"].notna().all(),
        "images_aligned": len(inventory_df) == len(image_paths),
    }

    for name, value in checks.items():
        print(f"{name}: {value}")

    ready = (
        checks["inventory_rows"] > 0
        and checks["image_files"] > 0
        and checks["titles_present"]
        and checks["prices_valid"]
        and checks["images_aligned"]
    )

    print("\nReady to run:", "YES" if ready else "NO")
    return ready


ready = preflight_check(selected_inventory, image_paths)


## 13. Run the Automation

> **Recommended first test:** keep `NUM_LISTINGS = 1`.

Only run this cell after manually confirming that the browser is logged in and that the selected test data is correct.


In [ ]:
if ready:
    results = create_facebook_listings(
        driver=driver,
        inventory_df=selected_inventory,
        image_paths=image_paths,
    )
    display(results)
else:
    print("Preflight check failed. Fix the inputs before running automation.")


## 14. Results Summary

The results table makes it easy to see how many listings succeeded and which listings need troubleshooting.


In [ ]:
if "results" in globals():
    total = len(results)
    successful = int((results["status"] == "Success").sum())
    failed = int((results["status"] == "Failed").sum())

    print(f"Total processed: {total}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")

    if failed:
        display(results.loc[results["status"] == "Failed"])


## 15. Close the Browser

Run this cell when the automation session is finished.


In [ ]:
def close_browser(driver):
    """Close the Selenium browser safely."""
    if driver is not None:
        try:
            driver.quit()
            print("Browser closed.")
        except WebDriverException:
            print("Browser was already closed.")


# Uncomment when finished:
# close_browser(driver)


## 16. Limitations and Future Improvements

### Current limitations

- Marketplace selectors can change when the website UI changes.
- Facebook may require additional verification or security checks.
- The notebook is designed for supervised use rather than unattended credential handling.
- Category, condition, tags, and description are currently configured as defaults.

### Future improvements

1. **Configuration-driven listings** — move category, condition, description, and tags into Excel.
2. **GUI application** — create a simple interface for selecting the workbook and image folder.
3. **Analytics dashboard** — track processed listings, success rate, failures, and execution time.
4. **AI-assisted descriptions** — generate descriptions from structured product information.
5. **Database integration** — replace the Excel-only workflow with a persistent inventory database.
6. **Multi-platform architecture** — reuse the same inventory data layer with platform-specific adapters.


## 17. Portfolio Takeaway

This project demonstrates practical skills in:

- Python programming
- Data ingestion and validation
- Browser automation with Selenium
- Workflow design
- Exception handling
- Spreadsheet-driven automation
- Reusable function design

### Portfolio title

**Marketplace Listing Automation System**

### One-line description

> A Python and Selenium automation workflow that converts spreadsheet inventory data into structured Marketplace listings through supervised browser automation.
